In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('data/tea_demand_dataset.csv', parse_dates=['week_start_date'])
df = df.sort_values(['grade', 'week_start_date']).reset_index(drop=True)

print(f"Loaded: {df.shape}")
print(f"Grades: {df['grade'].unique()}")

Loaded: (1040, 17)
Grades: ['BOP' 'BOPF' 'Dust' 'OP' 'Pekoe']


In [2]:
for lag in [1, 2, 4, 8, 13, 26, 52]:
    df[f'demand_lag{lag}w'] = df.groupby('grade')['demand_kg'].transform(
        lambda x: x.shift(lag)
    )

In [3]:
for window in [4, 12, 26]:
    df[f'demand_rollmean{window}w'] = df.groupby('grade')['demand_kg'].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).mean()
    )
    df[f'demand_rollstd{window}w'] = df.groupby('grade')['demand_kg'].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).std()
    )

In [4]:
df['week_sin'] = np.sin(2 * np.pi * df['week_of_year'] / 52)
df['week_cos'] = np.cos(2 * np.pi * df['week_of_year'] / 52)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['quarter'] = df['week_start_date'].dt.quarter

In [5]:
df = pd.get_dummies(df, columns=['grade'], prefix='grade')

In [6]:
df_clean = df.dropna().reset_index(drop=True)
print(f"Rows after cleaning: {len(df_clean)} (removed {len(df)-len(df_clean)} rows)")

Rows after cleaning: 780 (removed 260 rows)


In [7]:
df_clean.to_csv('data/tea_demand_processed.csv', index=False)
print("Saved processed dataset.")

Saved processed dataset.
